# Team SentiMix — ABSA for Hindi-English Code-Mixed E-commerce Reviews
**Starter Colab Notebook**

This notebook sets up the full pipeline skeleton matching Objectives 1–5 in your proposal:
1. Data collection / loading
2. Preprocessing (cleaning, normalization, tokenization)
3. Aspect extraction (rule-based baseline)
4. Sentiment classification (lexicon baseline + IndicBERT/MuRIL fine-tune)
5. Evaluation

Run cells top to bottom. Replace the dataset loading cell with whichever source you finalize.


In [ ]:
# ---- 1. Environment setup ----
!pip install -q transformers datasets evaluate scikit-learn indic-nlp-library emoji kagglehub
import pandas as pd
import numpy as np
import re
import emoji
import torch
print("GPU available:", torch.cuda.is_available())


## 2. Load dataset

Pick ONE of the options below depending on what you've finalized.
- Option A: Kaggle datasets (via `kagglehub`, no manual download needed)
- Option B: Upload your own scraped/annotated CSV

See the chat message for the actual dataset links + citations to put in your report.


In [ ]:
# ---- Option A: Kaggle dataset via kagglehub ----
import kagglehub

# Example: Meesho Multilingual Review Dataset (English-Marathi-Hindi code-mixed, e-commerce)
path = kagglehub.dataset_download("asmitapawar2005/meesho-multilingual-review-dataset")
print("Downloaded to:", path)

import os
for f in os.listdir(path):
    print(f)


In [ ]:
# ---- Option B: Upload your own CSV (manual annotation / scraped data) ----
from google.colab import files
uploaded = files.upload()  # choose your CSV
fname = list(uploaded.keys())[0]
df = pd.read_csv(fname)
df.head()


In [ ]:
# ---- Load into a standard dataframe ----
# Adjust column names to match whichever dataset you chose above.
# Expected minimal columns: 'review_text', 'rating' (or 'sentiment')

df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))  # adjust filename as needed
print(df.shape)
df.head()


## 3. Filter for genuinely code-mixed reviews

Central to your novelty claim — keep only reviews that actually mix Hindi (Romanized) and English,
not pure-English or pure-Devanagari text.


In [ ]:
# Simple heuristic Hindi (Romanized) word list — expand this as you inspect the data
HINDI_ROMAN_WORDS = set([
    "hai","nahi","bahut","accha","acha","achha","bekar","sahi","paisa","paise",
    "product","mast","badhiya","theek","kharab","bura","bura","kaam","pasand",
    "zyada","kam","lagta","hoga","mila","milta","dikkat","problem","bilkul"
])

def is_code_mixed(text):
    if not isinstance(text, str):
        return False
    tokens = re.findall(r"[a-zA-Z']+", text.lower())
    has_hindi = any(t in HINDI_ROMAN_WORDS for t in tokens)
    has_english = any(t not in HINDI_ROMAN_WORDS and len(t) > 2 for t in tokens)
    return has_hindi and has_english

df["is_code_mixed"] = df["review_text"].apply(is_code_mixed)
code_mixed_df = df[df["is_code_mixed"]].reset_index(drop=True)
print(f"Kept {len(code_mixed_df)} / {len(df)} reviews as code-mixed")
code_mixed_df.head()


## 4. Preprocessing pipeline (Objective 2)

Cleaning + normalization. Keep emojis (sentiment signal) instead of stripping them.


In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)             # URLs
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)              # baddddd -> badd
    text = emoji.demojize(text, delimiters=(" :", ": "))    # keep emoji meaning as text
    text = re.sub(r"\s+", " ", text).strip()
    return text

code_mixed_df["clean_text"] = code_mixed_df["review_text"].apply(clean_text)
code_mixed_df[["review_text", "clean_text"]].head()


In [ ]:
# Basic tokenization (swap for indic-nlp-library tokenizer once you've tested it on samples)
def tokenize(text):
    return re.findall(r"[\w']+|[.,!?;]", text)

code_mixed_df["tokens"] = code_mixed_df["clean_text"].apply(tokenize)
code_mixed_df["tokens"].head()


## 5. Aspect extraction — rule-based baseline (Objective 3)

Start here before moving to a fine-tuned BIO-tagging model. Curate the aspect vocabulary
per product category as you look at real examples.


In [ ]:
ASPECT_VOCAB = {
    "battery": ["battery", "backup", "charge", "charging"],
    "camera": ["camera", "photo", "picture", "selfie"],
    "price": ["price", "paisa", "paise", "cost", "expensive", "sasta", "mehenga"],
    "delivery": ["delivery", "shipping", "deliver", "late", "time pe"],
    "quality": ["quality", "build", "material", "finish"],
    "display": ["display", "screen", "resolution"],
}

def extract_aspects(tokens):
    text = " ".join(tokens).lower()
    found = []
    for aspect, keywords in ASPECT_VOCAB.items():
        if any(kw in text for kw in keywords):
            found.append(aspect)
    return found

code_mixed_df["aspects"] = code_mixed_df["tokens"].apply(extract_aspects)
code_mixed_df[["clean_text", "aspects"]].head(10)


## 6. Sentiment baseline — lexicon-based (Objective 4, Model 1)

Your comparison baseline, in the spirit of Paper 4 (Kavitha, 2022) from your literature survey.


In [ ]:
POSITIVE_WORDS = set(["accha","acha","achha","badhiya","mast","sahi","good","great","love","best","nice","awesome"])
NEGATIVE_WORDS = set(["bekar","kharab","bura","bad","worst","poor","problem","dikkat","waste","late"])

def lexicon_sentiment(text):
    tokens = re.findall(r"[a-zA-Z']+", text.lower())
    pos = sum(t in POSITIVE_WORDS for t in tokens)
    neg = sum(t in NEGATIVE_WORDS for t in tokens)
    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    return "neutral"

code_mixed_df["lexicon_sentiment"] = code_mixed_df["clean_text"].apply(lexicon_sentiment)
code_mixed_df["lexicon_sentiment"].value_counts()


## 7. Sentiment model — fine-tuning IndicBERT / MuRIL (Objective 4, Model 2)

This cell assumes you've built a labeled CSV with columns:
`text, aspect, sentiment` (sentiment ∈ {positive, negative, neutral}), from your manual annotation.

Framing: sentence-pair classification → `(review_text, aspect_term) → sentiment_label`


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

MODEL_NAME = "ai4bharat/indic-bert"  # alt: "google/muril-base-cased"
LABELS = {"negative": 0, "neutral": 1, "positive": 2}

# Load your manually annotated aspect-sentiment CSV here
# annotated_df = pd.read_csv("annotated_aspect_sentiment.csv")
# annotated_df["label"] = annotated_df["sentiment"].map(LABELS)

# train_df, test_df = train_test_split(annotated_df, test_size=0.15, stratify=annotated_df["label"], random_state=42)
# train_df, val_df   = train_test_split(train_df, test_size=0.176, stratify=train_df["label"], random_state=42)  # ~15% of total

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    return tokenizer(batch["text"], batch["aspect"], truncation=True, padding="max_length", max_length=128)

# train_ds = Dataset.from_pandas(train_df).map(preprocess, batched=True)
# val_ds   = Dataset.from_pandas(val_df).map(preprocess, batched=True)
# test_ds  = Dataset.from_pandas(test_df).map(preprocess, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
#     compute_metrics=compute_metrics,
# )
# trainer.train()


## 8. Evaluation (Objective 5)

Overall metrics + per-aspect breakdown + code-mixed vs pure-language subset comparison.


In [ ]:
# from sklearn.metrics import classification_report
# preds = trainer.predict(test_ds)
# y_pred = np.argmax(preds.predictions, axis=-1)
# y_true = test_df["label"].values
# print(classification_report(y_true, y_pred, target_names=list(LABELS.keys())))

# Per-aspect F1: group test_df by 'aspect' column and run classification_report on each subset
# Code-mixed vs pure-language: reuse the is_code_mixed() filter from Section 3 on the test set


## Next steps
- Finalize dataset (Section 2) and re-run the code-mixed filter
- Do the manual annotation pass (aspect + sentiment) — needed before Section 7 will actually run
- Swap the placeholder ASPECT_VOCAB / lexicon with your annotated categories
- Once trained, export metrics + confusion matrix for the progress report
